# Smartphone Addiction - GPU Stacking Pipeline

XGBoost + CatBoost + LightGBM, 5-fold OOF stacking with a logistic-regression meta-learner.
Fold-level checkpoints, 2-seed bagging, IterativeImputer fitted inside each fold.

Full code explanation: `NOTEBOOK_EXPLAINED.md`. Setup guide: `KAGGLE_SETUP.md`.


In [ ]:

import os, gc, glob, pickle, subprocess, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.special import logit
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")

SEED = 42
SEEDS = [42, 2024]
N_SPLITS = 5
ESR = 300
MAX_HOURS = 5.5
IMP_ITER = 20

TARGET = "addicted_label"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
NUM_COLS = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
            "work_study_hours", "sleep_hours", "notifications_per_day",
            "app_opens_per_day", "weekend_screen_time"]

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CKPT_DIR = f"{WORK}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
np.random.seed(SEED)

In [ ]:

def has_cuda():
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

def lgb_gpu_works():
    if not has_cuda():
        return False
    try:
        lgb.LGBMClassifier(n_estimators=2, device="gpu", verbose=-1).fit(
            np.random.rand(64, 4), np.random.randint(0, 2, 64))
        return True
    except Exception:
        return False

CUDA = has_cuda()
GPU = {"xgb": CUDA, "cat": CUDA, "lgb": lgb_gpu_works()}
N_JOBS = os.cpu_count()
print(f"cuda={CUDA}  gpu={GPU}  cores={N_JOBS}")

In [ ]:

def find_data():
    known = ["/kaggle/input/competitions/playground-series-s6e8",
             "/kaggle/input/playground-series-s6e8", "../data", "data"]
    cands = [f"{d}/train.csv" for d in known]
    cands += sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
    for f in cands:
        d = os.path.dirname(f)
        if os.path.exists(f) and os.path.exists(f"{d}/test.csv"):
            return d
    seen = sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]
    raise FileNotFoundError("train.csv + test.csv not found. /kaggle/input holds:\n" + "\n".join(seen))

DATA = find_data()
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")

y = train.pop(TARGET).values
test_ids = test["id"].values
X = train.drop(columns="id")
X_test = test.drop(columns="id")[X.columns]

for d in (X, X_test):
    for c in NUM_COLS:
        d[c] = d[c].astype("float32")

for c in CAT_COLS:
    levels = sorted(set(X[c].dropna()) | set(X_test[c].dropna()))
    X[c] = pd.Categorical(X[c], categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)

print(f"train {X.shape}  test {X_test.shape}  base rate {y.mean():.4f}  nan {X.isna().sum().sum():,}")

In [ ]:

IMP_COLS = [c + "_imp" for c in NUM_COLS]

def add_imputed(X_fit, frames):
    imp = IterativeImputer(max_iter=IMP_ITER, random_state=SEED).fit(X_fit[NUM_COLS])
    print(f"  imputer n_iter={imp.n_iter_}/{IMP_ITER}"
          + ("  <-- did not converge" if imp.n_iter_ >= IMP_ITER else ""))
    out = []
    for d in frames:
        d = d.copy()
        d[IMP_COLS] = imp.transform(d[NUM_COLS]).astype("float32")
        out.append(d)
    return out

def to_catboost(d):
    d = d.copy()
    for c in CAT_COLS:
        d[c] = d[c].astype(object).fillna("missing").astype(str)
    return d

In [ ]:

def make_xgb(seed):
    return xgb.XGBClassifier(
        n_estimators=30000, learning_rate=0.02, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
        reg_lambda=2.0, eval_metric="auc", early_stopping_rounds=ESR,
        enable_categorical=True, tree_method="hist",
        device="cuda" if GPU["xgb"] else "cpu",
        n_jobs=N_JOBS, random_state=seed,
    )

def make_cat(seed):
    return cb.CatBoostClassifier(
        iterations=30000, learning_rate=0.03, depth=8, l2_leaf_reg=6.0,
        eval_metric="AUC", random_seed=seed, verbose=False,
        allow_writing_files=False,
        task_type="GPU" if GPU["cat"] else "CPU",
    )

def make_lgb(seed):
    p = dict(
        n_estimators=10000, learning_rate=0.03, num_leaves=96,
        min_child_samples=50, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=2.0, objective="binary",
        random_state=seed, n_jobs=N_JOBS, verbose=-1,
    )
    if GPU["lgb"]:
        p.update(device="gpu", max_bin=255)
    return lgb.LGBMClassifier(**p)


def fit_xgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

def fit_cat(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=CAT_COLS,
          early_stopping_rounds=ESR, use_best_model=True, verbose=False)
    return m

def fit_lgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[lgb.early_stopping(ESR, verbose=False)])
    return m

MODELS = {
    "xgb": (make_xgb, fit_xgb, "num"),
    "cat": (make_cat, fit_cat, "cat"),
    "lgb": (make_lgb, fit_lgb, "num"),
}
NAMES = list(MODELS)


def best_iter(m):
    for a in ("best_iteration", "best_iteration_"):
        v = getattr(m, a, None)
        if isinstance(v, (int, np.integer)):
            return int(v)
    try:
        return int(m.get_best_iteration())
    except Exception:
        return None

def tree_cap(m):
    p = m.get_params()
    return p.get("n_estimators") or p.get("iterations")

In [ ]:

def new_state():
    return {"oof": np.zeros((len(X), len(NAMES)), dtype="float64"),
            "mask": np.zeros(len(X), dtype=bool),
            "test_sum": np.zeros((len(X_test), len(NAMES)), dtype="float64"),
            "done": [], "scores": {n: [] for n in NAMES}}

def load_state():
    for d in [CKPT_DIR] + sorted(glob.glob("/kaggle/input/**/checkpoints", recursive=True)):
        f = sorted(glob.glob(f"{d}/ckpt_fold*.pkl"))
        if f:
            s = pickle.load(open(f[-1], "rb"))
            print(f"resumed {f[-1]}  folds done: {s['done']}")
            return s
    return new_state()

state = load_state()

In [ ]:

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
t_start = time.time()
ran = 0

for fold, (tr, va) in enumerate(skf.split(X, y)):
    if fold in state["done"]:
        continue
    elapsed = time.time() - t_start
    if ran and elapsed + elapsed / ran > MAX_HOURS * 3600:
        print(f"next fold would exceed {MAX_HOURS}h - stopping with {len(state['done'])} folds done")
        break

    t0 = time.time()
    ytr, yva = y[tr], y[va]
    Xtr, Xva, Xte = add_imputed(X.iloc[tr], [X.iloc[tr], X.iloc[va], X_test])
    frames = {"num": (Xtr, Xva, Xte), "cat": tuple(to_catboost(d) for d in (Xtr, Xva, Xte))}

    for j, name in enumerate(NAMES):
        make, fit, kind = MODELS[name]
        A, B, C = frames[kind]
        pv = np.zeros(len(va))
        pt = np.zeros(len(X_test))
        bis, cap = [], None
        for s in SEEDS:
            m = fit(make(s), A, ytr, B, yva)
            pv += m.predict_proba(B)[:, 1] / len(SEEDS)
            pt += m.predict_proba(C)[:, 1] / len(SEEDS)
            bis.append(best_iter(m))
            cap = tree_cap(m)
            del m
            gc.collect()
        state["oof"][va, j] = pv
        state["test_sum"][:, j] += pt
        state["scores"][name].append(roc_auc_score(yva, pv))
        hit = cap and any(b is not None and b >= cap - 1 for b in bis)
        flag = f"  <-- HIT CAP {cap}, NOT CONVERGED" if hit else ""
        print(f"  fold {fold + 1} {name:4s} AUC={state['scores'][name][-1]:.5f}  "
              f"best_iter={bis}  ({time.time() - t0:.0f}s){flag}")

    state["mask"][va] = True
    state["done"].append(fold)
    ran += 1
    pickle.dump(state, open(f"{CKPT_DIR}/ckpt_fold{fold}.pkl", "wb"))
    del Xtr, Xva, Xte, frames
    gc.collect()
    print(f"fold {fold + 1}/{N_SPLITS} done in {time.time() - t0:.0f}s -> checkpoint saved")

print(f"\ntotal {(time.time() - t_start) / 60:.1f} min over {len(state['done'])} folds")

In [ ]:

assert state["done"], "no folds completed"

mask = state["mask"]
oof = state["oof"][mask]
y_oof = y[mask]
test_p = state["test_sum"] / len(state["done"])

for j, n in enumerate(NAMES):
    print(f"{n:4s} OOF AUC {roc_auc_score(y_oof, oof[:, j]):.5f}   folds {np.round(state['scores'][n], 5)}")
print(f"mean OOF AUC {roc_auc_score(y_oof, oof.mean(1)):.5f}")

In [ ]:

def L(p):
    return logit(np.clip(p, 1e-6, 1 - 1e-6))

meta = LogisticRegression(C=1.0, max_iter=1000)
honest = cross_val_predict(meta, L(oof), y_oof, cv=5, method="predict_proba")[:, 1]
print(f"stack OOF AUC {roc_auc_score(y_oof, honest):.5f}  (meta cross-validated)")

meta.fit(L(oof), y_oof)
print("meta weights:", dict(zip(NAMES, meta.coef_[0].round(3))))

In [ ]:

pred = meta.predict_proba(L(test_p))[:, 1]

sub = pd.DataFrame({"id": test_ids, TARGET: pred})
assert len(sub) == len(X_test), f"expected {len(X_test)} rows, got {len(sub)}"
assert sub["id"].is_unique
assert sub[TARGET].notna().all()
assert sub[TARGET].between(0, 1).all()
assert sub[TARGET].nunique() > 2

sub.to_csv(f"{WORK}/submission.csv", index=False)
np.savez(f"{WORK}/oof_stack.npz", oof=state["oof"], mask=state["mask"],
         test=test_p, names=np.array(NAMES))

print(f"{len(sub):,} rows  mean {pred.mean():.4f}  (train base rate {y.mean():.4f})")
if abs(pred.mean() - y.mean()) > 0.03:
    print("WARNING: mean prediction far from base rate - inspect before submitting")
print(f"folds used: {len(state['done'])}/{N_SPLITS}")
sub.head()